We now have a minimal implementation of CAD operations. It's time to test our new API against a provider.

We'll port over the Blender implementation and test running it with the new API.

In [ ]:
from codetocad.adapters.blender import *

In [ ]:
!uv pip install -e "..[blender]"

## Installing CodeToCAD inside Blender

Call the code cell below at least once to install CodeToCAD. 

After that, you can run scripts inside Blender using `run()`.

In [ ]:
from codetocad.adapters.blender.cli.blender_cli import (
    install_codetocad_in_blender,
    set_blender_executable_path,
)

set_blender_executable_path("/Applications/Blender.app/Contents/MacOS/Blender")

install_codetocad_in_blender()

## Running scripts in Blender

The new API allows you to run code inside Blender straight from this Jupyter notebook or a python script. 

No need to install an addon anymore, simplifying UX and allowing you to focus on just modeling!

In [ ]:
def create_cube():
    from codetocad.cli.config import get_temp_stl_export_path
    from codetocad.adapters.blender.blender_actions.import_export import export_object
    from codetocad.adapters.blender.blender_actions.objects import get_object
    from codetocad.adapters.blender.blender_definitions import (
        BlenderObjectPrimitiveTypes,
    )
    from codetocad.adapters.blender.blender_actions.objects import add_primitive

    primitive_to_add = BlenderObjectPrimitiveTypes.cylinder

    name = primitive_to_add.default_name_in_blender()

    add_primitive(primitive_to_add, dimensions=[1, 1, 1])

    assert export_object(
        get_object(name), str(get_temp_stl_export_path())
    ), "Export failed."

In [ ]:
from codetocad.adapters.blender.cli.blender_cli import run

run(create_cube, background=False)

### Visualize using Open3D

In [8]:
from platform import system

import open3d as o3d

from codetocad.cli.config import get_temp_stl_export_path

if system() != "Darwin":
    from open3d.web_visualizer import draw
else:
    # The web visualizer is not available on macOS.
    from open3d.visualization import draw

mesh = o3d.io.read_triangle_mesh(str(get_temp_stl_export_path()))
mesh.paint_uniform_color([0.5, 0.5, 0.5])  # Gray color

draw(mesh)

## Taking the new API for a spin

It's time to test the new API in Blender.

In [ ]:
def create_cube():
    """Create a simple cube using the new Blender CAD implementation."""
    try:
        # Import the Blender CAD implementation
        from codetocad.adapters.blender.cad.part import Part

        print("🎯 Creating a cube using Blender CAD implementation...")

        # Create a cube part using the preset
        cube = Part.preset.cube(2, 2, 2)
        cube.set_name("test_cube")

        print(f"✅ Successfully created cube: {cube}")
        print(f"   Cube name: {cube.name}")
        print(f"   Blender object: {cube.get_blender_object()}")

        return cube

    except Exception as e:
        print(f"❌ Error creating cube: {e}")
        import traceback

        traceback.print_exc()
        return None


run(create_cube, background=False)

In [ ]:
def test_vertex_edge():
    """Test basic vertex and edge creation."""
    try:
        from codetocad.adapters.blender.cad.vertex import Vertex
        from codetocad.adapters.blender.cad.edge import Edge

        print("🔧 Testing vertex and edge creation...")

        # Create vertices
        v1 = Vertex(0, 0, 0, name="origin")
        v2 = Vertex(1, 1, 1, name="corner")

        print(f"   Created vertex 1: {v1}")
        print(f"   Created vertex 2: {v2}")

        # Create edge
        edge = Edge(v1, v2, name="test_edge")
        print(f"   Created edge: {edge}")
        print(f"   Edge length: {edge.length()}")

        return edge

    except Exception as e:
        print(f"❌ Error in vertex/edge test: {e}")
        import traceback

        traceback.print_exc()
        return None


run(test_vertex_edge, background=False)

In [ ]:
def test_assembly():
    """Test assembly creation with multiple parts."""
    try:
        from codetocad.adapters.blender.cad.assembly import Assembly
        from codetocad.adapters.blender.cad.part import Part

        print("🏗️ Testing assembly creation...")

        # Create an assembly
        assembly = Assembly("test_assembly")

        # Create parts
        cube = Part.preset.cube(1, 1, 1)
        cube.set_name("assembly_cube")

        cylinder = Part.preset.cylinder(0.5, 2)
        cylinder.set_name("assembly_cylinder")

        # Add parts to assembly
        assembly.add(cube)
        assembly.add(cylinder)

        print(f"   Created assembly: {assembly}")
        print(f"   Assembly parts: {len(assembly.parts)}")

        return assembly

    except Exception as e:
        print(f"❌ Error in assembly test: {e}")
        import traceback

        traceback.print_exc()
        return None


run(test_vertex_edge, background=False)